In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
import os
load_dotenv()

# Conditional Chains with Review analyzer


True

In [2]:
# initialize the model

repo_id = "meta-llama/Llama-3.1-8B-Instruct"

llm = HuggingFaceEndpoint(
    repo_id= repo_id,
    max_new_tokens= 50,
    temperature= 0.5,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

model = ChatHuggingFace(llm = llm)

# We will use same model for everything

e:\AI-LLMs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class ReviewLLMOut(BaseModel):
    sentiment : Literal["pos","neg"] = Field(...,description="sentiment analysis of the text. ")

In [10]:
# Using Structured Output Parser

from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda

parser = PydanticOutputParser(pydantic_object=ReviewLLMOut)
parser2 = StrOutputParser()

prompt_sentiment = PromptTemplate(
    template = "classify the sentiment of following text : {text} into pos and neg in following format : {format_instructions}",
    input_variables= ["text"],
    partial_variables={"format_instructions":parser.get_format_instructions()}
    )

prompt_positive = PromptTemplate(
    template= "Generate positive response to the following positive review : {text}",
    input_variables=["text"]
)

prompt_negative = PromptTemplate(
    template= "Generate feedback, summarizing improvements for the following negative review : {text}",
    input_variables=["text"]
)

text = """Pretty rude staff, especially the host and hostess.  Super smug and seem delighted to turn you away even when you try reserving weeks in advance.  Even 2 weeks out, apparently, there is no room unless you want to eat lunch at 3 pm.  They act as if they had a Michelin star and that their food is enough to keep them in business.  I can’t wait to see this yuppy unoriginal trash close down in a year, like every other one in NYC before it."""

sentiment_chain = prompt_sentiment | model | parser

conditional_chain = RunnableBranch(
    (lambda x:x.sentiment=="pos", RunnableLambda(lambda x: text) | prompt_positive | model | parser2),
    (lambda x:x.sentiment=="neg", RunnableLambda(lambda x: text) | prompt_negative | model | parser2),
    RunnableLambda( lambda x:"sentiment was not detected, or invalid! ")
)

total_chain = sentiment_chain | conditional_chain


response = total_chain.invoke({"text":text})


print(response)

Here's a summary of potential improvements and feedback that the restaurant could use to address the customer's concerns:

**Key Issues:**

1. **Rude staff**: The host and hostess were perceived as smug and dismissive, making customers feel unwelcome.
2. **Inflexible reservation policy**: The restaurant was unwilling to accommodate reservations, even when made weeks in advance, and offered limited options for alternative times.
3. **Attitude and entitlement**: The staff seemed to believe that the restaurant's quality and reputation were sufficient to attract customers, without making an effort to provide good service.

**Recommendations for Improvement:**

1. **Staff training**: Provide training to staff on customer service and hospitality, emphasizing the importance of being welcoming and accommodating to all customers.
2. **Flexible reservation policy**: Consider implementing a more flexible reservation policy, allowing for more flexibility in scheduling and accommodating customer pr